# Baseline modeling

**GOAL:**

- Train multiple models
- compare their performance and metrics
- select the promising model

In [1]:
# Imports

import pandas as pd
import numpy as np

In [2]:
# load X_train, X_val, X_test, y_train, y_val, y_test

X_train = pd.read_csv(r'..\Data\processed\modeling\X_train.csv')
X_val = pd.read_csv(r'..\Data\processed\modeling\X_val.csv')
X_test = pd.read_csv(r'..\Data\processed\modeling\X_test.csv')

y_train = pd.read_csv(r'..\Data\processed\modeling\y_train.csv')
y_val = pd.read_csv(r'..\Data\processed\modeling\y_val.csv')
y_test = pd.read_csv(r'..\Data\processed\modeling\y_test.csv')

## 1: Baseline model

- linear regression

**comparison models**

- ridge regression
- random forest
- XG boost

**Comparison**

- compare by RMSE Rsqt

In [3]:
# 1.1 Baseline model selcetion (Linear regression)
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train)
lr_y_pred = lr.predict(X_val)

In [4]:
# 1.2 Modle evaluation (Error metric and Eval metric)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error

# Regression error metrics
lr_mae = mean_absolute_error(y_val, lr_y_pred)
lr_mse = mean_squared_error(y_val, lr_y_pred)
lr_rmse = root_mean_squared_error(y_val, lr_y_pred)
print('Error metrics for log transformed rent\n')
print(f'MAE of Linear Regression is: {mean_absolute_error(y_val, lr_y_pred)}')
print(f'MSE of Linear Regression is: {lr_mse}')
print(f'RMSE of Linear Regression is: {lr_rmse}')

# regression evaluation metric
lr_r2 = r2_score(y_val, lr_y_pred)

print(f'\nR2 Score of Linear Regression is: {lr_r2}')

# actual rent metrics

actual_y_val = np.expm1(y_val)
actual_lr_y_pred = np.expm1(lr_y_pred)

# Regression error metrics
lr_mae = mean_absolute_error(actual_y_val, actual_lr_y_pred)
lr_mse = mean_squared_error(actual_y_val, actual_lr_y_pred)
lr_rmse = root_mean_squared_error(actual_y_val, actual_lr_y_pred)

print('\nError metrics for actual rent\n')
print(f'MAE of Linear Regression is: {lr_mae}')
print(f'MSE of Linear Regression is: {lr_mse}')
print(f'RMSE of Linear Regression is: {lr_rmse}')

# regression evaluation metric

print(f'\nR2 Score of Linear Regression is: {lr_r2}')

Error metrics for log transformed rent

MAE of Linear Regression is: 0.1994128873538294
MSE of Linear Regression is: 0.09182113494594465
RMSE of Linear Regression is: 0.3030200240016238

R2 Score of Linear Regression is: 0.3344733140396472

Error metrics for actual rent

MAE of Linear Regression is: 1754.580646621875
MSE of Linear Regression is: 19910211.309875313
RMSE of Linear Regression is: 4462.085981900765

R2 Score of Linear Regression is: 0.3344733140396472


# Baseline Model Report

## MAE - Error Metric
- Mean Absolute Error in actual price,
    - tells me that my regression model's predictions are **on average, ~1754 rs** away from the actual rent

## RMSE - Error Metric
- RMSE tells a gap,
    ```md
    RMSE = 4462
    MAE  = 1754
    Gap  = 2708  ← this is large
    ```
    - This gap means my model has some bad predictions (outliers), that pulls up RMSE score
    - Most predictions are around ₹1,754 off (MAE)
    - But a few are severely bad — pulling RMSE up to ₹4,462
    - The gap (₹2,708) just tells us how much those outliers are skewing the score

## R2 Score - Eval metric

- R2 Score tells me that my baseline model (Linear Regression), can explains about ~33% of variation in the rent
    - so Since R2 score was low (below 50% means) the model is weak

- ** why R2 score is low?**
    - Linear regression can not predict non linear predictions

# Final Observation: Linear Regression Baseline

---

| Metric | Score | Indication | Better | Decision |
|--------|-------|------------|--------|----------|
| **MAE** | ₹1,754 | On average, every prediction is ₹1,754 away from the actual rent | ⬇️ Lower = Better | Train another model and compare |
| **RMSE > MAE** | Gap = ₹2,708 | Some predictions are severely bad (outliers) | ⬇️ Lower = Better | Train another model and compare |
| **R² Score** | 0.33 | Linear Regression model's prediction is weak — ~67% of rent variation unexplained | ⬆️ Higher = Better | Train another model (tree-based) to improve |

---

> **Verdict:** Linear Regression is not good enough. Move to tree-based models (Random Forest / XGBoost) to capture non-linear rent patterns.

In [16]:
lr.coef_

array([[-0.07099695,  0.40381681,  0.44220164,  0.00606746, -0.00652418,
        -0.11000773,  0.13723976, -0.00303758,  0.25176793,  0.02357802,
        -0.09637525,  0.00417878,  0.15154267,  0.02307527, -0.04958345,
         0.03007933,  0.1287062 , -0.26377551,  0.01675561,  0.003946  ,
         0.10853681,  0.07348781, -0.11116017,  0.06455939,  0.06289336,
        -0.0180705 , -0.04482286,  0.04378457, -0.05731983,  0.06514942,
        -0.05161416, -0.03545369,  0.01008463,  0.02536906]])

---
# Train More Models to comapre against baseline
---

## **Ok now before trying tree based models, I'm going to try regularization with the linear model**

### L2 Regularization (Ridge Regression)

- Since i have many featues and corelating with each (some), linear regression may gave too much importance to some features
- The goal is to penalize those coefficients and potentially reduce the variance or overfitting 


In [17]:
X_train.head()

,latitude,longitude,locality,transit_score,lifestyle_score,occupancy,deposit,attached_bathroom,food_included,mess,...,gender_BOTH,gender_FEMALE,gender_MALE,parking_Bike,parking_Bike and Car,parking_Car,parking_No Parking,available_for_Anyone,available_for_Student,available_for_Working Professional
0,12.913368,80.228812,8.935934,6.3,6.3,2.0,8.294300,1,1,1,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,12.979819,80.242495,8.901274,8.0,7.9,2.0,8.006701,0,1,0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,12.989589,80.248599,8.886307,7.9,8.3,0.0,10.283669,0,0,1,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,12.926202,80.111700,8.876267,8.2,9.6,2.0,7.601402,0,1,0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,13.053022,80.213763,8.693086,6.7,6.3,2.0,10.308986,0,0,0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [19]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# before train a ridge model, i need to scale the features, cuz ridge is sensitive to features scale
# so basically ridge is used with standard scalar

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)

ridge.fit(X_train_scaled, y_train)

ridge_y_pred = ridge.predict(X_val_scaled)

print(f'MAE of Ridge is: {mean_absolute_error(y_val, ridge_y_pred)}')
print(f'MSE of Ridge is: {mean_squared_error(y_val, ridge_y_pred)}')
print(f'R2 Score of Ridge is: {r2_score(y_val, ridge_y_pred)}')

MAE of Ridge is: 0.199342230234515
MSE of Ridge is: 0.09179847693517831
R2 Score of Ridge is: 0.3346375410536637


In [27]:
for alpha in [0, 1,10, 100, 200, 10.5]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    pred = ridge.predict(X_val_scaled)

    print(
        alpha,
        r2_score(y_val, pred),
        mean_absolute_error(y_val, pred),
        root_mean_squared_error(y_val, pred),
    )

0 0.32501407380814473 0.20144136751383282 0.30516586282979413
1 0.3346375410536637 0.199342230234515 0.3029826347089521
10 0.3356823502043269 0.1989498506920994 0.3027446566004236
100 0.3342582094154708 0.19967247051705744 0.3030689895381113
200 0.3263372002669429 0.20154787526609194 0.30486661874354587
10.5 0.33572514546676935 0.19893437935122074 0.3027349050563396


## Ridge Regression result:

- alpha = ~10.0 gives the best result = 0.3357
- still similar to the linear regression, no drastical improvements


**Takeaway**: Ridge isn't dramatically better than Linear Regression for this dataset. That's actually a useful modeling finding.